# Silver — GDP Indicators

Cleans and enriches Bronze GDP data: adds a proper date column for each quarter and rounds values.

**Source:** `bronze.statistics_iceland_gdp`  
**Output:** `silver.gdp_indicators` (Delta table)  
**Date rule:** Q1 → Jan 1, Q2 → Apr 1, Q3 → Jul 1, Q4 → Oct 1  

In [ ]:
df = spark.read.format("delta").table("bronze.statistics_iceland_gdp")

print(f"Bronze rows: {df.count()}")
df.show()

In [ ]:
df_silver = spark.sql("""
    SELECT
        quarter,
        year,
        q,
        TO_DATE(CONCAT_WS('-', year, (q * 3 - 2), '1'), 'yyyy-M-d') AS date,
        metric,
        ROUND(value, 2) AS value,
        CURRENT_TIMESTAMP() AS processed_at
    FROM bronze.statistics_iceland_gdp
    ORDER BY year, q
""")

if df_silver.isEmpty():
    raise ValueError("[silver_statistics] No rows returned - halting.")

In [ ]:
df_silver.createOrReplaceTempView("v_gdp_silver")

if not spark.catalog.tableExists("silver.gdp_indicators"):
    df_silver.write.format("delta").saveAsTable("silver.gdp_indicators")
else:
    spark.sql("""
        MERGE INTO silver.gdp_indicators AS target
        USING v_gdp_silver AS source
        ON target.year = source.year AND target.q = source.q
        WHEN MATCHED THEN UPDATE SET
            target.quarter      = source.quarter,
            target.date         = source.date,
            target.metric       = source.metric,
            target.value        = source.value,
            target.processed_at = source.processed_at
        WHEN NOT MATCHED THEN INSERT (quarter, year, q, date, metric, value, processed_at)
        VALUES (source.quarter, source.year, source.q, source.date, source.metric, source.value, source.processed_at)
    """)

print(f"Saved to silver.gdp_indicators - {spark.table('silver.gdp_indicators').count()} rows")